# 構造化された結果の取得方法

## 1、with_structured_output を使用

例：

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# .envファイルから環境変数を読み込む
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model = init_chat_model(
    model="openai/gpt-4o-mini",
    model_provider="openai",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

In [2]:
from pydantic import BaseModel, Field
from rich import print

class Movie(BaseModel):
    """映画情報"""
    title: str = Field(description="映画タイトル")
    year: int = Field(description="公開年")
    director: str = Field(description="監督")
    rating: float = Field(description="評価（10点満点）")


structured_model = model.with_structured_output(Movie,include_raw=True)
response = structured_model.invoke("映画『インターステラー』を紹介してください")

print(type(response))
print(response)   # 出力結果には元の AIMessage が含まれる

<class 'dict'>

{
    'raw': AIMessage(
        content='{"title":"インターステラー","year":2014,"director":"クリストファー・ノーラン","rating":8.6}',
        additional_kwargs={
            'parsed': Movie(title='インターステラー', year=2014, director='クリストファー・ノーラン', rating=8.6),
            'refusal': None
        },
        response_metadata={
            'token_usage': {
                'completion_tokens': 32,
                'prompt_tokens': 182,
                'total_tokens': 214,
                'completion_tokens_details': {
                    'accepted_prediction_tokens': None,
                    'audio_tokens': 0,
                    'reasoning_tokens': 0,
                    'rejected_prediction_tokens': None,
                    'image_tokens': 0
                },
                'prompt_tokens_details': {
                    'audio_tokens': 0,
                    'cached_tokens': 0,
                    'cache_write_tokens': 0,
                    'video_tokens': 0
                },
                'cost': 4.65e-05,
                'is_byok': False,
                'cost_details': {
                    'upstream_inference_cost': 4.65e-05,
                    'upstream_inference_prompt_cost': 2.73e-05,
                    'upstream_inference_completions_cost': 1.92e-05
                }
            },
            'model_provider': 'openai',
            'model_name': 'openai/gpt-4o-mini',
            'system_fingerprint': 'fp_27599ce29d',
            'id': 'gen-1785657368-6tMMLPiAOq5s0kHDZzur',
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='lc_run--019fc179-1b5d-72a1-8199-f262ec3a2b1e-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 182,
            'output_tokens': 32,
            'total_tokens': 214,
            'input_token_details': {'audio': 0, 'cache_read': 0},
            'output_token_details': {'audio': 0, 'reasoning': 0}
        }
    ),
    'parsed': Movie(title='インターステラー', year=2014, director='クリストファー・ノーラン', rating=8.6),
    'parsing_error': None
}

## 2、出力パーサーを使用（参考）

In [4]:

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 1. プロンプトテンプレートを作成
prompt_template = ChatPromptTemplate.from_messages([
    ("system","ユーザーの質問に回答し、必ず title（映画タイトル）と year（公開年）を含む JSON オブジェクトを出力すること"),
    ("human","質問：{question}")
])

# 2. モデルの初期化
# .envファイルから環境変数を読み込む
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

model = init_chat_model(
    model="openai/gpt-4o-mini",
    model_provider="openai",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

# 3. 構造を定義
class Movie(BaseModel):
    """映画情報"""
    title: str = Field(description="映画タイトル")
    year: int = Field(description="公開年")

# 4. 出力パーサーを作成
parser = JsonOutputParser(pydantic_object=Movie)

# 5. チェーンを作成
chain = prompt_template | model | parser

# 6. 呼び出し（辞書を返す）
# response = chain.invoke({"question": "映画『インセプション』を紹介してください"})

response = parser.invoke(model.invoke(prompt_template.invoke({"question": "映画『インセプション』を紹介してください"})))


print(type(response))
print(response)

<class 'dict'>

{
    'title': 'インセプション',
    'year': 2010,
    'description': 
'『インセプション』は、クリストファー・ノーランが監督したサイエンスフィクション・アクション映画です。夢の中に入り込
み、他人のアイデアを盗むことができる特殊な能力を持つ泥棒ドミニク・コブ（レオナルド・ディカプリオ）が、逆に他人の夢
にアイデアを植え付ける任務を請け負う様子を描いています。複雑なストーリー展開と圧巻のビジュアルが特徴で、観る者を引
き込む作品です。'
}